# 卷积神经网络 CNN：图像识别的王者
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：06_深度学习 | 源文件：卷积神经网络_CNN.py | 核心功能：卷积层、池化层、特征提取、图像分类

## 概述

CNN 是处理网格数据（图像、音频）的专用神经网络。核心思想：用卷积核（小窗口）在输入上滑动，提取局部特征。通过多层卷积，从低级特征（边缘）逐步构建高级特征（物体部件）。

与 MLP 的关键区别：**局部连接**（只看小区域）+ **权值共享**（同一卷积核到处用）= 参数大幅减少。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
# --- 导入库 ---
from torchvision.utils import make_grid
import numpy as np

## 数学原理

### 1. 卷积操作

**代码对应**：`nn.Conv2d(in_channels, out_channels, kernel_size, padding)`。

2D 离散卷积：

$$(\mathbf{f} * \mathbf{g})(i, j) = \sum_m\sum_n f(m, n) \cdot g(i-m, j-n)$$

输出特征图：$O[c', i, j] = \sum_c\sum_m\sum_n W[c', c, m, n] \cdot I[c, i+m, j+n] + b[c']$

### 2. 输出尺寸计算

$$H_{\text{out}} = \left\lfloor\frac{H_{\text{in}} + 2 \cdot \text{padding} - \text{kernel\_size}}{\text{stride}} + 1\right\rfloor$$

**代码对应**：`padding=1, kernel_size=3, stride=1` 时输出尺寸不变。

### 3. 参数共享与局部连接

全连接层参数量：$H \times W \times C \times H' \times W' \times C'$

卷积层参数量：$K \times K \times C_{\text{in}} \times C_{\text{out}}$（与输入尺寸无关）

参数共享使 CNN 对平移具有等变性：如果输入平移，输出也相应平移。

### 4. 池化层

MaxPool：$O[i, j] = \max_{(m,n) \in R_{ij}} I[m, n]$

池化增加平移不变性，降低空间分辨率，减少计算量。

### 5. 感受野

第 $l$ 层神经元的感受野大小递推：

$$RF_l = RF_{l-1} + (k_l - 1) \times \prod_{i=1}^{l-1}s_i$$

深层神经元的感受野覆盖更大区域，能捕捉更全局的特征。

### 6. Dropout

训练时以概率 $p$ 随机将神经元输出置零：

$$h_i' = \begin{cases} 0 & \text{with prob } p \\ h_i / (1-p) & \text{with prob } 1-p \end{cases}$$

Dropup 是一种隐式集成（$2^n$ 个子网络的平均）。

### 1. 加载 MNIST 数据

运行 1. 加载 MNIST 数据 的代码，观察算法在该环节的行为。

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST 均值和标准差
])

train_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST("./data", train=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

### 2. 定义 CNN 模型

运行 2. 定义 CNN 模型 的代码，观察算法在该环节的行为。

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 卷积层: 1×28×28 → 32×14×14 → 64×7×7
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 1→32 通道
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28→14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 32→64 通道
# --- 继续 ---
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14→7
        )
        # 全连接层
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
# --- 继续 ---
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"=== CNN 模型结构 ===")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}")

### 3. 查看卷积核

运行 3. 查看卷积核 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 卷积层信息 ===")
for name, param in model.named_parameters():
    if "weight" in name and "features" in name:
        print(f"  {name}: shape={param.shape}")

### 4. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 训练 ===")
n_epochs = 5
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
# --- 赋值/配置 ---
    correct = 0
    total = 0
    for batch_X, batch_y in train_loader:
        output = model(batch_X)
        loss = criterion(output, batch_y)
# --- 继续 ---
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (output.argmax(1) == batch_y).sum().item()
# --- 继续 ---
        total += batch_y.size(0)

    # 测试
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
# --- 继续 ---
            output = model(batch_X)
            test_correct += (output.argmax(1) == batch_y).sum().item()
            test_total += batch_y.size(0)

    print(f"  Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, "
          f"Train Acc={correct/total:.4f}, Test Acc={test_correct/test_total:.4f}")

### 5. 特征图可视化

运行 5. 特征图可视化 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 卷积特征提取示意 ===")
# 展示中间层输出形状
sample = train_dataset[0][0].unsqueeze(0)  # 取一个样本
model.eval()
with torch.no_grad():
    feat1 = model.features[0](sample)  # 第一层卷积输出
    feat2 = model.features[3](model.features[2](model.features[1](feat1)))  # 第二层
# --- 输出结果 ---
print(f"输入:       {sample.shape}")
print(f"Conv1 输出: {feat1.shape}")
print(f"Conv2 输出: {feat2.shape}")

### 6. CNN 要点

运行 6. CNN 要点 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== CNN 关键组件 ===")
print("1. 卷积层 (Conv2d): 局部连接 + 权值共享，提取局部特征")
print("2. 池化层 (MaxPool2d): 降采样，增加平移不变性")
print("3. 全连接层: 将特征图映射到类别输出")
print("4. ReLU: 非线性激活")
# --- 输出结果 ---
print("5. Dropout: 正则化，防过拟合")

print("\n=== CNN 要点 ===")
print("- 卷积核大小: 3×3 最常用（VGG 风格）")
print("- 通道数: 逐层增加（1→32→64→128）")
print("- 每次池化分辨率减半，通道数翻倍")
print("- 参数共享使 CNN 比全连接网络参数量少得多")
# --- 输出结果 ---
print("- 适合：图像分类、目标检测、语义分割等视觉任务")

## 关键代码解释

### 卷积层

```python
nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
```

padding=1 保持空间尺寸不变。输出通道数 = 卷积核数量。

### 典型 CNN 架构

```python
Conv2d -> ReLU -> MaxPool -> Conv2d -> ReLU -> MaxPool -> Flatten -> Linear -> Output
```

## 注意事项

1. **输入格式**：(batch, channels, height, width)
2. **感受野**：深层卷积核看到的区域更大
3. **数据增强**：CNN 对平移、旋转敏感，需要数据增强

## 总结与延伸

以上代码展示了 **卷积神经网络 CNN** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **深度可分离卷积**：MobileNet 的核心，大幅减少参数
- **空洞卷积**：扩大感受野而不增加参数
- **注意力机制**：Vision Transformer 是否能取代 CNN
